# API Beauty Match

Ce notebook contient le site FastAPI et le modèle de recommandation fondé sur les identifiants d'ingrédients. Exécutez d'abord la cellule de l'application, puis celle de démarrage du serveur pour ouvrir le site à l'adresse `http://127.0.0.1:8000/`. Lorsque vous avez terminé, exécutez la cellule d'arrêt du serveur.

Le fichier `app.py` reste disponible comme point d'entrée pour le déploiement sur un service d'hébergement.


In [ ]:
# Exécutez cette commande pip dans l'environnement du projet pour éviter les conflits de dépendances.
# %pip install -r requirements.txt

In [ ]:
from pathlib import Path
from typing import Dict, List, Optional, Set

import numpy as np
import pandas as pd
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from fastapi.staticfiles import StaticFiles
from pydantic import BaseModel, Field
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:

BASE_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
if not (BASE_DIR / "data" / "products.csv").is_file():
    BASE_DIR = next(
        (
            parent
            for parent in (Path.cwd(), *Path.cwd().parents)
            if (parent / "data" / "products.csv").is_file()
        ),
        BASE_DIR,
    )

DATA_DIR = BASE_DIR / "data"
STATIC_DIR = BASE_DIR / "static"
CSV_PATH = DATA_DIR / "products.csv"
PRODUCT_INGREDIENTS_PATH = DATA_DIR / "product_ingredients.csv"
PRODUCT_IMAGES_DIR = DATA_DIR / "images"
METADATA_FEATURES = ("category", "subcategory", "product_type")

app = FastAPI(
    title="The Beauty API",
    description="API de recommandation de produits utilisant le jeu de données products.csv.",
    version="1.2.0",
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)
app.mount("/images", StaticFiles(directory=PRODUCT_IMAGES_DIR), name="product-images")


def load_and_prepare_data(
    products_path: str,
    product_ingredients_path: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if not Path(products_path).is_file():
        raise FileNotFoundError(f"Fichier CSV des produits introuvable : {products_path}")
    if not Path(product_ingredients_path).is_file():
        raise FileNotFoundError(f"Fichier CSV des ingrédients des produits introuvable : {product_ingredients_path}")

    df = pd.read_csv(products_path, keep_default_na=False)
    ingredient_data = pd.read_csv(
        product_ingredients_path,
        usecols=["product_id", "ingredient_id", "label_name"],
        dtype=str,
        keep_default_na=False,
    )

    required_columns = [
        "id",
        "brand",
        "name",
        "category",
        "subcategory",
        "ingredients_raw",
        "origin",
        "product_type",
        "benefits",
        "directions",
        "warnings",
        "contains_fragrance",
        "contains_drying_alcohol",
        "contains_parabens",
        "contains_sulfates",
        "contains_silicones",
    ]

    for col in required_columns:
        if col not in df.columns:
            df[col] = ""

    text_columns = [
        "brand",
        "name",
        "category",
        "subcategory",
        "ingredients_raw",
        "origin",
        "product_type",
        "benefits",
        "directions",
        "warnings",
    ]
    for col in text_columns:
        df[col] = df[col].fillna("").astype(str)

    df["id"] = pd.to_numeric(df["id"], errors="raise").astype(int)

    bool_columns = [
        "contains_fragrance",
        "contains_drying_alcohol",
        "contains_parabens",
        "contains_sulfates",
        "contains_silicones",
    ]
    for col in bool_columns:
        if col in df.columns:
            df[col] = df[col].map(
                lambda value: value if isinstance(value, bool) else str(value).strip().lower() == "true"
            )

    ingredient_data["product_id"] = pd.to_numeric(ingredient_data["product_id"], errors="raise").astype(int)
    ingredient_data["ingredient_id"] = ingredient_data["ingredient_id"].str.strip()
    ingredient_data = ingredient_data[ingredient_data["ingredient_id"] != ""]
    return df, ingredient_data


def build_feature_rows(
    products: pd.DataFrame,
    ingredient_data: pd.DataFrame,
) -> List[Dict[str, float]]:
    ingredients_by_product: Dict[int, Set[str]] = (
        ingredient_data.groupby("product_id")["ingredient_id"]
        .agg(lambda ingredient_ids: set(ingredient_ids))
        .to_dict()
    )
    feature_rows = []
    for product in products.itertuples(index=False):
        features = {
            f"ingredient:{ingredient_id}": 1.0
            for ingredient_id in ingredients_by_product.get(product.id, set())
        }
        for column in METADATA_FEATURES:
            value = str(getattr(product, column, "")).strip().casefold()
            if value:
                features[f"{column}:{value}"] = 1.0
        feature_rows.append(features)

    return feature_rows


df, ingredient_data = load_and_prepare_data(CSV_PATH, PRODUCT_INGREDIENTS_PATH)
feature_vectorizer = DictVectorizer()
feature_matrix = feature_vectorizer.fit_transform(build_feature_rows(df, ingredient_data))
cosine_sim = cosine_similarity(feature_matrix, feature_matrix)


class RecommendationRequest(BaseModel):
    selected_ids: List[int] = Field(..., min_items=1, max_items=5, description="De 1 à 5 identifiants de produits")
    excluded_ingredients: List[str] = Field(default_factory=list, description="Ingrédients à exclure")
    top_n: int = Field(default=3, ge=1, le=20)


class ProductResponse(BaseModel):
    id: int
    brand: str = ""
    name: str = ""
    image_name: Optional[str] = None
    category: str = ""
    subcategory: str = ""
    size: Optional[str] = None
    benefits: Optional[str] = None
    origin: str = ""
    product_type: str = ""
    ingredients_raw: str = ""
    contains_fragrance: bool = False
    contains_drying_alcohol: bool = False
    contains_parabens: bool = False
    contains_sulfates: bool = False
    contains_silicones: bool = False
    similarity_score: Optional[float] = None


@app.get("/", include_in_schema=False)
def home():
    return FileResponse(STATIC_DIR / "index.html")


@app.get("/products", response_model=List[ProductResponse], tags=["Catalogue"])
def get_all_products():
    if df.empty:
        raise HTTPException(status_code=500, detail="Le jeu de données n'est pas chargé.")
    return df.to_dict(orient="records")


@app.post("/recommend", response_model=List[ProductResponse], tags=["Recommandation"])
def get_recommendations(request: RecommendationRequest):
    if df.empty:
        raise HTTPException(status_code=500, detail="Le jeu de données n'est pas chargé.")

    indices = df[df["id"].isin(request.selected_ids)].index.tolist()
    if not indices:
        raise HTTPException(status_code=404, detail="Aucun produit valide trouvé pour les identifiants sélectionnés.")

    mean_sim_scores = np.mean(cosine_sim[indices], axis=0)

    results_df = df.copy()
    results_df["similarity_score"] = np.round(mean_sim_scores, 3)
    results_df = results_df[~results_df["id"].isin(request.selected_ids)]

    if request.excluded_ingredients:
        for ingredient in request.excluded_ingredients:
            clean_ing = str(ingredient).strip()
            if clean_ing:
                excluded_product_ids = ingredient_data.loc[
                    ingredient_data["label_name"].str.contains(
                        clean_ing,
                        case=False,
                        na=False,
                        regex=False,
                    ),
                    "product_id",
                ].unique()
                results_df = results_df[~results_df["id"].isin(excluded_product_ids)]

    results_df = results_df.sort_values(by="similarity_score", ascending=False).head(request.top_n)
    return results_df.to_dict(orient="records")


## Démarrer le site

Exécutez cette cellule une fois pour démarrer le site en arrière-plan.


In [ ]:
import threading
import time
import uvicorn

server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="info"))
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()
for _ in range(100):
    if server.started:
        break
    if not server_thread.is_alive():
        raise RuntimeError("Le serveur web s'est arrêté avant d'avoir fini de démarrer.")
    time.sleep(0.1)
if not server.started:
    raise TimeoutError("Le serveur web n'a pas démarré dans les 10 secondes.")
print("Le site est accessible à l'adresse http://127.0.0.1:8000/")


## Arrêter le site

Exécutez cette cellule pour arrêter proprement le serveur.


In [ ]:
if "server" in globals() and server_thread.is_alive():
    server.should_exit = True
    server_thread.join(timeout=10)
    if server_thread.is_alive():
        raise TimeoutError("Le serveur web ne s'est pas arrêté dans les 10 secondes.")
    print("Le serveur du site est arrêté.")
else:
    print("Le serveur du site n'est pas en cours d'exécution.")
